In [9]:

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

## **Routers vs Abtraction Layers**:
Routers like open router help to route the request to different providers. Abstraction layers like Langchain and LiteLLM  provide aditional feartues as abstraction.

In [11]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content)) 

Why did the aspiring LLM engineer bring coffee, a debugger, and a validation set to class?

Because becoming an expert takes caffeine, patience, and something to tell you when your model is confidently wrong.

## LiteLLM 

In [12]:
from litellm import completion
response = completion(model="gpt-5-nano", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM engineering student bring a map to the data center? To navigate the long tail of training data. 

Need more options? I can give you a few more one-liners.

In [13]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 23
Output tokens: 2033
Total tokens: 2056
Total cost: 0.0814 cents


## Prompt Caching using Lite LLM

In [14]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


### Asking questions without sending context

In [15]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [16]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks "Where is my father?" (referring to Polonius), Claudius replies:

**"Dead."**

Gertrude quickly adds, **"But not by him,"** clarifying immediately that Hamlet is not the one who killed him (or, more accurately, that Claudius was not the one responsible, as he is trying to deflect blame).

### Now sending hamlet as context

In [17]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [18]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In the text provided, when Laertes enters with his followers and demands "Where is my father?" the King replies:

**"Dead."**

This occurs in Act IV, Scene V.

In [19]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output tokens: 40
Cached tokens: None
Total cost: 1.3362 cents


In [20]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In the provided text of *Hamlet*, the scene you are referring to is **Act IV, Scene V**. When Laertes bursts into the court and demands his father, the King replies:

> "Dead."

Immediately after, the Queen adds, "But not by him!"

In [21]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output tokens: 57
Cached tokens: 49124
Total cost: 0.2335 cents


When you repeatedly send the **same or very similar input context** within a short period, the provider can reuse work it has already done instead of processing the entire prompt again. This significantly reduces the cost of subsequent requests.

-   A large prompt containing the **entire text of Hamlet** is sent to the model.
-   The **first request** processes all ~53,000 input tokens and costs about **0.5 cents**.
-   The **second request**, made shortly afterward with the same context, uses **cached tokens** (about 52,200 of the 53,000 input tokens), reducing the cost to roughly **0.1 cents**—about **5× cheaper**.

How to maximize cache hitsThe beginning of the prompt should remain **identical** across requests.
Good structure:
```
[Large static context]
[User question]
```
Bad structure:
```
[User question]
[Large static context]
```

Similarly, dynamic values such as the current date or timestamp should be placed **near the end** of the prompt rather than at the beginning. If the first part of the prompt changes, prompt caching may not work.

Provider differences

-   **OpenAI:** Prompt caching is generally automatic when the initial portion of the prompt matches exactly.
-   **Anthropic:** Caching must be explicitly enabled ("prime the cache"). Priming costs about **25% more**, but subsequent cached requests are about **10× cheaper**.
-   **Gemini:** Supports both implicit (automatic) and explicit caching modes.
    
Why it matters

Prompt caching is most valuable when:

-   You repeatedly send large documents (manuals, books, codebases, knowledge bases).
-   Multiple users query the same reference material.
-   You want to reduce API costs while maintaining the same quality of responses.

The text also highlights that tools like **LiteLLM** make caching easy to observe by reporting:

-   Total input/output tokens
-   Cached input tokens
-   Cost per API call

This makes it practical to monitor API spending and optimize the economics of production AI applications.